# ============================================================
# STEP 0 — SCRIPT OVERVIEW (Markdown cell content)
# ============================================================
# This script:
# 1. Loads an EPC dataset from a CSV file
# 2. Creates a lookup table mapping MAIN_HEAT_CODE → HP_UPGRADE (0/1)
# 3. Applies the mapping to create a new column called HP_UPGRADE
# 4. Checks results and basic value counts for validation
# 5. (Optional) Saves the updated dataset back to CSV
#
# HP_UPGRADE meaning:
# - 0 = heat pump system already present (no upgrade needed)
# - 1 = upgrade candidate (non-heat pump or legacy system)
# ============================================================

In [1]:
# ============================================================
# STEP 1 — IMPORT LIBRARIES
# Description: Load required Python libraries
# ============================================================

import pandas as pd

In [2]:
# ============================================================
# STEP 2 — LOAD DATASET
# Description: Read EPC dataset from CSV file
# ============================================================

file_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_lat_long_4_dec.csv"

df = pd.read_csv(file_path)

# Quick check
df.head()

,BUILDING_REFERENCE_NUMBER,OSG_REFERENCE_NUMBER,ADDRESS1,ADDRESS2,ADDRESS3,POSTCODE,LATITUDE,LONGITUDE,ORIENTATION,ORIENTATION_ESPR_ROT,...,PANELS_ALONG_WIDTH,POSSIBLE_PV_PANELS,WINDOW_AREA_SINGLE,ASPECT_RATIO,WINDOW_H,WINDOW_W,EXISTING_PV_PANELS,MODELED_PV,PV_UPGRADE,ACH_NATURAL
0,1001829067,122009933.0,19 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.0557,-4.2233,100,170,...,2,0,1.790829,1.75,1.011598,1.770297,0,0,1,1.05
1,1001951858,122010025.0,GLENVIEW,FINTRY,GLASGOW,G63 0XJ,56.0528,-4.2206,180,90,...,3,8,3.189657,2.25,1.190641,2.678942,0,8,1,1.01
2,1001829071,122009868.0,22 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.0555,-4.2237,100,170,...,3,2,2.044886,1.75,1.080975,1.891706,0,2,1,0.82
3,1234761001,122009968.0,1 MENZIES TERRACE,FINTRY,GLASGOW,G63 0YJ,56.0584,-4.2248,150,120,...,3,8,3.295329,2.25,1.210203,2.722956,0,8,1,1.22
4,1001991633,122009892.0,50 MAIN STREET,FINTRY,GLASGOW,G63 0XF,56.0547,-4.2283,150,120,...,5,21,2.763486,2.25,1.108249,2.493560,0,21,1,0.84


In [3]:
# ============================================================
# STEP 3 — CREATE LOOKUP DICTIONARY
# Description: Map MAIN_HEAT_CODE values to HP_UPGRADE (0/1)
# ============================================================

hp_upgrade_map = {
    # HP already present → 0
    "air_source_heat_pump_radiators_electric_2": 0,
    "air_source_heat_pump_radiators_electric_3": 0,
    "air_source_heat_pump_radiators_electric_4": 0,
    "air_source_heat_pump_radiators_electric_5": 0,
    "air_source_heat_pump_underfloor_electric_4": 0,
    "air_source_heat_pump_underfloor_electric_5": 0,
    "ground_source_heat_pump_radiators_electric_3": 0,
    "ground_source_heat_pump_underfloor_electric_3": 0,
    "ground_source_heat_pump_underfloor_electric_4": 0,
    "ground_source_heat_pump_underfloor_electric_5": 0,

    # Non-HP systems → 1 (upgrade candidate)
    "boiler_radiators_dual_fuel_mineral_wood_3": 1,
    "boiler_radiators_electric_1": 1,
    "boiler_radiators_electric_3": 1,
    "boiler_radiators_lpg_1": 1,
    "boiler_radiators_lpg_2": 1,
    "boiler_radiators_mains_gas_4": 1,
    "boiler_radiators_oil_2": 1,
    "boiler_radiators_oil_3": 1,
    "boiler_radiators_wood_pellets_3": 1,
    "community_scheme_4": 1,
    "electric_storage_heaters_1": 1,
    "electric_storage_heaters_3": 1,
    "electric_storage_heaters_4": 1,
    "no_system_present_electric_heaters_assumed_1": 1,
    "room_heaters_dual_fuel_mineral_wood_2": 1,
    "room_heaters_electric_1": 1,
    "room_heaters_electric_3": 1
}

In [4]:
# ============================================================
# STEP 4 — CREATE HP_UPGRADE COLUMN
# Description: Apply mapping to MAIN_HEAT_CODE
# ============================================================

df["HP_UPGRADE"] = df["MAIN_HEAT_CODE"].map(hp_upgrade_map)

# Check for unmapped values (important QA step)
unmapped = df[df["HP_UPGRADE"].isna()]["MAIN_HEAT_CODE"].unique()
print("Unmapped MAIN_HEAT_CODE values:", unmapped)

Unmapped MAIN_HEAT_CODE values: []


In [5]:
# ============================================================
# STEP 5 — VALIDATION CHECKS
# Description: Inspect distribution of HP_UPGRADE values
# ============================================================

df["HP_UPGRADE"].value_counts(dropna=False)

HP_UPGRADE
1    80
0    37
Name: count, dtype: int64

In [6]:
# ============================================================
# STEP 6 — SAVE UPDATED DATASET (OPTIONAL)
# Description: Write updated CSV to disk
# ============================================================

output_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_hp_upgrade.csv"

df.to_csv(output_path, index=False)

print("Saved updated dataset to:", output_path)

Saved updated dataset to: /Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_hp_upgrade.csv
